# Stage 2B — MULTI-VIEW cross-plane MAE (the twist)

Both an axial AND a coronal slice of the same subject are 75%-masked; each view's visible tokens are the context for reconstructing the other. The encoder must internalize cross-plane 3-D anatomical consistency. Learned view embeddings keep the planes distinguishable. **Everything else is identical to Stage 2A** — same data, same seeds, same architecture, same schedule (paired comparison by design).

**Kaggle reality notes**
- Enable GPU + Internet (timm not needed here; Internet optional).
- `max_minutes=600` stops cleanly before the 12 h session limit; rerun the SAME
  notebook version (or a new version) and it **resumes** from `ckpt.pt`.
- Estimated wall time on T4 x2: single ≈ 3–5 h, cross ≈ 5–8 h for 60 epochs.


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.pretrain import run_pretrain

cfg = load_config("configs/pretrain_cross.yaml")
summary = run_pretrain(cfg)

In [ ]:
# ---- VERIFICATION ----
import json, pandas as pd
a = pd.DataFrame([json.loads(l) for l in open("outputs/pretrain_single/pretrain_single.jsonl")
                  if json.loads(l)["event"] == "epoch"])
b = pd.DataFrame([json.loads(l) for l in open("outputs/pretrain_cross/pretrain_cross.jsonl")
                  if json.loads(l)["event"] == "epoch"])
cmp = pd.DataFrame({"single": a.loss.values[:len(b)], "cross": b.loss.values})
print(cmp.tail(10).to_string(index=False))
print("final losses ->", cmp.iloc[-1].to_dict())

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 7))
latest = sorted(Path("outputs/pretrain_cross").glob("recon_samples_e*.png"))[-1]
ax.imshow(Image.open(latest)); ax.axis("off")
ax.set_title(f"cross-arm reconstruction grid — {latest.name}")
plt.tight_layout(); plt.show()